# Help Codes

## 1 Copy Unique Images: Folder A → Folder C
This notebook copies images from **Folder A** that are **not present in Folder B** into **Folder C**.

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

### Configure Your Folder Paths

In [ ]:
# Source folder
FOLDER_A = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Raw/Synthetic/template-c'
# Reference folder (images to exclude)
FOLDER_B = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Training/Normalized/Synthetic/template-c'
# Destination folder
FOLDER_C = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Raw/Evaluation/Synthetic-1/template-c'

COMPARE_MODE = 'filename'   # Change to 'hash' for content-based comparison

print(f'Folder A (source):      {FOLDER_A}')
print(f'Folder B (reference):   {FOLDER_B}')
print(f'Folder C (destination): {FOLDER_C}')
print(f'Compare mode: {COMPARE_MODE}')

### Run the Copy Script

In [ ]:
import os
import shutil
import hashlib
from pathlib import Path

# Supported image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.tif'}


def get_image_files(folder):
    """Return a list of image file paths in a folder (non-recursive)."""
    folder = Path(folder)
    return [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS]


def file_hash(filepath):
    """Compute MD5 hash of a file for content comparison."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def copy_unique_images(folder_a, folder_b, folder_c, compare_mode='filename'):
    folder_a = Path(folder_a)
    folder_b = Path(folder_b)
    folder_c = Path(folder_c)

    # Validate source folders
    if not folder_a.exists():
        raise FileNotFoundError(f'Folder A not found: {folder_a}')
    if not folder_b.exists():
        raise FileNotFoundError(f'Folder B not found: {folder_b}')

    # Create destination folder if needed
    folder_c.mkdir(parents=True, exist_ok=True)
    print(f'Folder C ready: {folder_c}\n')

    # Get images
    images_a = get_image_files(folder_a)
    images_b = get_image_files(folder_b)

    print(f'Found {len(images_a)} image(s) in Folder A')
    print(f'Found {len(images_b)} image(s) in Folder B')
    print(f'Comparing by: {compare_mode}\n')

    # Build lookup set for Folder B
    if compare_mode == 'hash':
        print('Computing hashes for Folder B...')
        b_lookup = {file_hash(f) for f in images_b}
        print(f'   Done. {len(b_lookup)} unique hash(es) in Folder B.\n')
    else:
        b_lookup = {f.name.lower() for f in images_b}

    # Copy images from A not in B
    copied, skipped = 0, 0

    for img in images_a:
        if compare_mode == 'hash':
            key = file_hash(img)
        else:
            key = img.name.lower()

        if key not in b_lookup:
            dest = folder_c / img.name
            # Handle filename conflicts in Folder C
            if dest.exists():
                stem, suffix = dest.stem, dest.suffix
                counter = 1
                while dest.exists():
                    dest = folder_c / f'{stem}_{counter}{suffix}'
                    counter += 1
            shutil.copy2(img, dest)
            print(f'Copied: {img.name} → {dest.name}')
            copied += 1
        else:
            print(f'Skipped (exists in B): {img.name}')
            skipped += 1

    print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'🎉 Done!')
    print(f'Copied:  {copied} image(s)')
    print(f'Skipped: {skipped} image(s)')
    print(f':  {folder_c}')


# ▶️ Run it!
copy_unique_images(FOLDER_A, FOLDER_B, FOLDER_C, compare_mode=COMPARE_MODE)